# Week 2 — The Vacuum World: Four Agent Architectures

**Lesson plan:** [`../weeks/week-02.md`](../weeks/week-02.md) · **Slides:** [`../slides/week-02/deck.md`](../slides/week-02/deck.md)

Live coding, 25 minutes. Build one environment, then run three agents against one
interface. The sequence matters: each agent fails in a way that motivates the next.

**Key idea to land:** internal state is what buys you the ability to *know you are
finished*.

## 1 · The environment

Two rooms. A percept is `(location, is_dirty_here)`. Note what the agent
**cannot** see: whether the *other* room is dirty.

In [1]:
class VacuumWorld:
    """Two rooms. Percept = (location, status_here)."""

    def __init__(self, dirt=(True, True), loc=0):
        self.dirt, self.loc, self.score, self.t = list(dirt), loc, 0, 0

    def percept(self):
        return (self.loc, self.dirt[self.loc])

    def step(self, action):
        if action == "Suck":
            self.dirt[self.loc] = False
        elif action == "Left":
            self.loc = 0
        elif action == "Right":
            self.loc = 1
        # performance measure: clean rooms, summed over time
        self.score += sum(not d for d in self.dirt)
        self.t += 1

    def __repr__(self):
        rooms = "".join("D" if d else "." for d in self.dirt)
        return f"[{rooms}] agent@{self.loc} t={self.t} score={self.score}"


w = VacuumWorld()
print(w, "| percept:", w.percept())

[DD] agent@0 t=0 score=0 | percept: (0, True)


## 2 · A simple reflex agent

Maps the current percept straight to an action. No memory of any kind.

In [2]:
def simple_reflex(percept):
    loc, dirty = percept
    if dirty:
        return "Suck"
    return "Right" if loc == 0 else "Left"


def run(agent, steps=20, dirt=(True, True), loc=0, trace=0):
    world = VacuumWorld(dirt, loc)
    for i in range(steps):
        action = agent(world.percept())
        if trace and i < trace:
            print(f"  {world}  ->  {action}")
        world.step(action)
    return world


w = run(simple_reflex, steps=8, trace=8)
print("\nfinal:", w)

  [DD] agent@0 t=0 score=0  ->  Suck
  [.D] agent@0 t=1 score=1  ->  Right
  [.D] agent@1 t=2 score=2  ->  Suck
  [..] agent@1 t=3 score=4  ->  Left
  [..] agent@0 t=4 score=6  ->  Right
  [..] agent@1 t=5 score=8  ->  Left
  [..] agent@0 t=6 score=10  ->  Right
  [..] agent@1 t=7 score=12  ->  Left

final: [..] agent@0 t=8 score=14


## 3 · Break the sensor — partial observability in four lines

Real sensors fail. Suppose the location sensor dies and reports `None`.

In [3]:
class BrokenSensorWorld(VacuumWorld):
    def percept(self):
        return (None, self.dirt[self.loc])   # location unknown


try:
    world = BrokenSensorWorld()
    action = simple_reflex(world.percept())   # dirt=True, so it sucks — fine
    world.step(action)
    print("after Suck:", world, "percept:", world.percept())
    action = simple_reflex(world.percept())   # now clean here; which way to move?
    print("action chosen:", action)
except Exception as e:
    print(f"{type(e).__name__}: {e}")

print("""
The agent cannot decide. With loc=None it has no basis for Left vs Right.
This is PARTIAL OBSERVABILITY, live, in four lines of change.
A reflex agent's competence is exactly as good as its sensors.""")

after Suck: [.D] agent@0 t=1 score=1 percept: (None, False)
action chosen: Left

The agent cannot decide. With loc=None it has no basis for Left vs Right.
This is PARTIAL OBSERVABILITY, live, in four lines of change.
A reflex agent's competence is exactly as good as its sensors.


## 4 · A model-based agent — add memory

The fix is internal state: track what has been cleaned, and where we believe we are.

> **Note — the belief is only as good as its starting assumption.** This agent
> *never reads the location from the percept* (`_, dirty = percept`). That is
> deliberate: it is exactly what lets it survive the broken sensor above, where
> location is `None`. The price is that it *assumes it starts in room 0*. In a
> world that starts the agent in room 1 it will clean that room, believe it is
> finished, and leave room 0 dirty — visible below as its lower score on the
> `loc=1` row. Memory buys you "I know I'm done"; it does not buy you a correct
> initial belief.

In [4]:
def model_based():
    state = {"loc": 0, "clean": [False, False]}

    def agent(percept):
        _, dirty = percept
        if dirty:
            state["clean"][state["loc"]] = True
            return "Suck"
        state["clean"][state["loc"]] = True
        if all(state["clean"]):
            return "NoOp"                  # knows it is done — reflex never does
        state["loc"] = 1 - state["loc"]
        return "Right" if state["loc"] == 1 else "Left"

    return agent


w = run(model_based(), steps=8, trace=8)
print("\nfinal:", w)

  [DD] agent@0 t=0 score=0  ->  Suck
  [.D] agent@0 t=1 score=1  ->  Right
  [.D] agent@1 t=2 score=2  ->  Suck
  [..] agent@1 t=3 score=4  ->  NoOp
  [..] agent@1 t=4 score=6  ->  NoOp
  [..] agent@1 t=5 score=8  ->  NoOp
  [..] agent@1 t=6 score=10  ->  NoOp
  [..] agent@1 t=7 score=12  ->  NoOp

final: [..] agent@1 t=8 score=14


## 5 · The comparison

Run both for 20 steps on every starting configuration and total the score.

In [7]:
configs = [((True, True), 0), ((True, True), 1),
           ((True, False), 0), ((False, True), 0),
           ((False, False), 0)]

print(f"{'start':<22} {'reflex':>8} {'model-based':>12}")
print("-" * 44)
tot_r = tot_m = 0
for dirt, loc in configs:
    r = run(simple_reflex, 20, dirt, loc).score
    m = run(model_based(), 20, dirt, loc).score
    tot_r += r
    tot_m += m
    label = f"dirt={dirt} loc={loc}"
    print(f"{label:<22} {r:>8} {m:>12}")
print("-" * 44)
print(f"{'TOTAL':<22} {tot_r:>8} {tot_m:>12}")

start                    reflex  model-based
--------------------------------------------
dirt=(True, True) loc=0       38           38
dirt=(True, True) loc=1       38           20
dirt=(True, False) loc=0       40           40
dirt=(False, True) loc=0       39           39
dirt=(False, False) loc=0       40           40
--------------------------------------------
TOTAL                       195          177


### What the numbers say

The model-based agent stops moving once the world is clean. The reflex agent
oscillates forever — and every pointless move is cost it can never recover.

> **Internal state is what buys you the ability to know you are finished.**

Notice this is not a smarter *rule*. It is a different *architecture*.

## 6 · ⚠️ You get what you ask for

Now a utility-based agent. Movement costs 1 point; a clean room is worth 1 per
step. Watch what a *rational* agent does with that specification.

In [6]:
def utility_agent(move_cost=1.0):
    state = {"loc": 0, "clean": [False, False]}

    def agent(percept):
        _, dirty = percept
        if dirty:
            state["clean"][state["loc"]] = True
            return "Suck"
        state["clean"][state["loc"]] = True
        other = 1 - state["loc"]
        # Expected gain from crossing vs. the cost of crossing.
        if not state["clean"][other] and move_cost < 1.0:
            state["loc"] = other
            return "Right" if other == 1 else "Left"
        return "NoOp"          # crossing is not worth it
    return agent


for cost in (0.5, 1.0):
    w = run(utility_agent(cost), steps=20, dirt=(False, True), loc=0)
    print(f"move_cost={cost}: {w}")

print("""
With move_cost=1.0 the agent RATIONALLY LEAVES THE OTHER ROOM DIRTY.
It is not broken. It is maximizing the utility function you wrote.

You have just built a specification-gaming agent. In week 14 this gets its
proper name: reward hacking / Goodhart's law / proxy misalignment.
Remember that you built one in week 2 before you knew the term.""")

move_cost=0.5: [..] agent@1 t=20 score=39
move_cost=1.0: [.D] agent@0 t=20 score=20

With move_cost=1.0 the agent RATIONALLY LEAVES THE OTHER ROOM DIRTY.
It is not broken. It is maximizing the utility function you wrote.

You have just built a specification-gaming agent. In week 14 this gets its
proper name: reward hacking / Goodhart's law / proxy misalignment.
Remember that you built one in week 2 before you knew the term.


## 7 · Exercise for the studio

1. Write a **goal-based** agent that plans a route rather than reacting.
2. Extend `VacuumWorld` to *n* rooms. Which agents still work? Which need
   rewriting? (This is the atomic → factored representation question from the
   slides, arriving as a refactor.)
3. Add a 10 % chance that `Suck` fails. Which architecture degrades most
   gracefully, and why?